# Amazon Bedrock AgentCore Runtime에서 Strands 에이전트와 Datadog LLM Observability 사용하기

## 개요

이 노트북에서는 Datadog LLM Observability를 연동한 Strands 에이전트를 Amazon Bedrock AgentCore Runtime에 배포하는 방법을 살펴봅니다. Amazon Bedrock Claude 모델을 사용하며, OpenTelemetry(OTEL)를 통해 텔레메트리 데이터를 Datadog으로 전송합니다.

## 주요 구성 요소

- **Strands Agents**: 기본 텔레메트리 지원이 포함된 LLM 기반 에이전트 구축용 Python 프레임워크
- **Amazon Bedrock AgentCore Runtime**: AWS에서 에이전트를 호스팅하고 확장하기 위한 관리형 런타임 서비스
- **Datadog LLM Observability**: GenAI 특화 트레이스 화면을 제공하는 LLM 애플리케이션 모니터링 플랫폼
- **OpenTelemetry**: 텔레메트리 데이터를 수집하고 내보내기 위한 업계 표준 프로토콜

## 아키텍처

에이전트는 컨테이너로 패키징되어 호출용 HTTP 엔드포인트를 제공하는 AgentCore Runtime에 배포됩니다. 텔레메트리 데이터는 Strands 에이전트에서 OTLP exporter를 거쳐 Datadog의 트레이스 엔드포인트로 직접 전달되어 모니터링과 디버깅에 사용됩니다. 이 구현에서는 Datadog을 사용하기 위해 AgentCore의 기본 ADOT 관측성을 비활성화합니다.

## 사전 요구 사항

- Python 3.10+
- Bedrock 및 AgentCore 권한이 구성된 AWS 자격 증명
- API 키가 있는 [Datadog](https://www.datadoghq.com/) 계정
- 로컬에 설치된 Docker
- 구성한 리전의 Amazon Bedrock Claude 모델에 대한 액세스 권한

In [ ]:
!pip install --force-reinstall -U -r requirements.txt

## AWS Credentials 구성

## Agent 구현

에이전트 파일(`strands_claude.py`)은 calculator 및 weather 도구를 갖춘 여행 assistant를 구현합니다. 주요 구성은 다음과 같습니다.
- **`DISABLE_ADOT_OBSERVABILITY=true`**: 자체 TracerProvider를 설정할 수 있도록 AgentCore의 기본 ADOT pipeline을 비활성화합니다([문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html)).
- **직접 OTLP 내보내기**: Datadog의 OTLP 엔드포인트로 트레이스를 전송하는 OpenTelemetry TracerProvider를 구성합니다.
- **`dd-otlp-source=llmobs`**: GenAI 전용 화면을 위해 트레이스를 [Datadog LLM Observability](https://docs.datadoghq.com/llm_observability/)로 라우팅합니다.
- **`OTEL_SEMCONV_STABILITY_OPT_IN=gen_ai_latest_experimental`**: Strands Agents에 필요한 OpenTelemetry v1.37+ GenAI semantic convention을 활성화합니다([문서](https://docs.datadoghq.com/llm_observability/instrumentation/otel_instrumentation/)).
- **자동 트레이스 내보내기**: 모든 에이전트 호출, 도구 호출, LLM 상호 작용이 자동으로 추적되어 Datadog으로 전송됩니다.

In [ ]:
%%writefile strands_claude.py
import os
import logging

logging.basicConfig(level=logging.ERROR, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(os.getenv("AGENT_RUNTIME_LOG_LEVEL", "INFO").upper())

# =============================================================================
# Datadog LLM Observability - OpenTelemetry 구성
# 다른 OpenTelemetry import보다 먼저 구성해야 함
# =============================================================================

# 자체 TracerProvider를 설정할 수 있도록 AgentCore의 기본 ADOT 비활성화
# 참고: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html
os.environ["DISABLE_ADOT_OBSERVABILITY"] = "true"

# strands-agents GenAI semantic convention(v1.37+)에 필요
os.environ["OTEL_SEMCONV_STABILITY_OPT_IN"] = "gen_ai_latest_experimental"

dd_api_key = os.environ.get("DD_API_KEY", "")
dd_site = os.environ.get("DD_SITE", "datadoghq.com")
service_name = os.environ.get("OTEL_SERVICE_NAME", "agentcore-datadog-sample")

if dd_api_key:
    from opentelemetry import trace
    from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import SimpleSpanProcessor
    from opentelemetry.sdk.resources import Resource

    resource = Resource.create({"service.name": service_name})
    exporter = OTLPSpanExporter(
        endpoint=f"https://trace.agent.{dd_site}/v1/traces",
        headers={"dd-api-key": dd_api_key, "dd-otlp-source": "llmobs"},
    )
    provider = TracerProvider(resource=resource)
    provider.add_span_processor(SimpleSpanProcessor(exporter))
    trace.set_tracer_provider(provider)
    logger.info("Datadog LLM Observability configured (service: %s)", service_name)
else:
    logger.warning("DD_API_KEY not set. Traces will not be sent to Datadog.")

# =============================================================================
# 에이전트
# =============================================================================

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator


def get_bedrock_model():
    region = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
    model_id = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
    return BedrockModel(
        model_id=model_id,
        region_name=region,
        max_tokens=1024,
    )


bedrock_model = get_bedrock_model()

system_prompt = """You are a helpful travel assistant. You can perform mathematical calculations 
and check weather information. Always provide helpful, accurate responses and use tools when appropriate."""


@tool
def weather():
    """Get current weather."""
    return "sunny and 72F"


app = BedrockAgentCoreApp()


def initialize_agent():
    """에이전트를 초기화합니다(텔레메트리는 모듈 수준에서 이미 구성됨)."""
    return Agent(
        model=bedrock_model,
        system_prompt=system_prompt,
        tools=[calculator, weather],
    )


@app.entrypoint
def strands_agent_bedrock(payload, context=None):
    """페이로드로 에이전트를 호출합니다."""
    user_input = payload.get("prompt", payload.get("text", payload.get("message", "Hello")))
    logger.info("[%s] User input: %s", getattr(context, 'session_id', 'local'), user_input)

    agent = initialize_agent()
    response = agent(user_input)
    return response.message['content'][0]['text']


if __name__ == "__main__":
    app.run()

### AgentCore Runtime 배포 구성

이제 starter toolkit을 사용하여 진입점, 앞서 생성한 실행 역할, requirements 파일로 AgentCore Runtime 배포를 구성합니다. 또한 시작할 때 Amazon ECR 리포지토리를 자동으로 생성하도록 starter toolkit을 구성합니다.

구성 단계에서는 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다. `bedrock_agentcore_starter_toolkit`으로 에이전트를 구성하면 AgentCore Observability가 기본으로 설정되므로, Datadog을 사용하려면 아래 설명과 같이 AgentCore Observability 구성을 제거해야 합니다.

<div style="text-align:left">
    <img src="../images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_datadog_llm_observability"
response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode="NO_MEMORY",
    disable_otel=True,
)
response

## AgentCore Runtime에 배포

Dockerfile이 준비되었으므로 에이전트를 AgentCore Runtime에 배포합니다. 이 과정에서 Amazon ECR 리포지토리와 AgentCore Runtime이 생성됩니다.

### Datadog 구성

Datadog LLM Observability로 트레이스를 전송하려면 다음 항목이 필요합니다.
- **Datadog API Key**: Datadog 계정의 Organization Settings → API Keys에서 가져옵니다.

`DD_API_KEY`가 제공되면 에이전트 코드(`strands_claude.py`)가 모든 OTLP 설정을 자동으로 구성합니다.
- **엔드포인트**: `https://trace.agent.{DD_SITE}/v1/traces`
- **헤더**: `dd-api-key={key},dd-otlp-source=llmobs`
- **Semantic Conventions**: `gen_ai_latest_experimental`(LLM Observability용)

**다른 Datadog 리전**을 사용하려면 시작 시 env_vars에 `DD_SITE`를 설정합니다.
- US1: `datadoghq.com`(기본값)
- US3: `us3.datadoghq.com`
- US5: `us5.datadoghq.com`
- EU1: `datadoghq.eu`
- AP1: `ap1.datadoghq.com`

<div style="text-align:left">
    <img src="../images/launch.png" width="75%"/>
</div>

In [ ]:
# Datadog 구성
datadog_api_key = "<datadog-api-key>"  # Datadog API key로 교체

launch_result = agentcore_runtime.launch(
    env_vars={
        "DD_API_KEY": datadog_api_key,
        "DD_SITE": "datadoghq.com",  # 다른 Datadog region에서는 변경
        "OTEL_SERVICE_NAME": "agentcore-datadog-sample",
        "DISABLE_ADOT_OBSERVABILITY": "true",  # AgentCore의 기본 observability 비활성화
    }
)
launch_result

## 배포 상태 확인

호출하기 전에 런타임이 준비될 때까지 기다립니다.

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

### AgentCore Runtime 호출

마지막으로 페이로드를 사용하여 AgentCore Runtime을 호출합니다.

<div style="text-align:left">
    <img src="../images/invoke.png" width="75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke(
    {"prompt": "What is 25 + 17 and what's the weather like for a trip to Paris?"}
)

In [ ]:
from IPython.display import Markdown, display

display(Markdown("".join(invoke_response["response"])))

## Datadog LLM Observability에서 트레이스 확인

에이전트를 호출하면 몇 분 안에 Datadog에 트레이스가 표시됩니다.

1. [Datadog LLM Observability](https://app.datadoghq.com/llm/traces)로 이동합니다.
2. `ml_app:agentcore-datadog-sample`을 검색합니다.

트레이스에는 다음 정보가 포함됩니다.
- 전체 요청/응답 컨텍스트를 포함한 에이전트 호출 세부 정보
- 실행 시간을 포함한 도구 호출(calculator, weather)
- 지연 시간 및 토큰 사용량을 포함한 모델 상호 작용
- Prompt 및 completion 내용

<div style="text-align:left">
    <img src="../images/llm-obs-example.png" width="75%"/>
</div>

### Datadog LLM Observability 기능

Datadog LLM Observability는 GenAI 애플리케이션에 특화된 화면을 제공합니다.

- **Trace Explorer**: 하나의 타임라인에서 prompt/response 내용, 도구 호출, 모델 상호 작용이 포함된 end-to-end 에이전트 트레이스 확인
- **Token Usage Tracking**: 비용과 성능을 최적화할 수 있도록 모델별 input/output 토큰 사용량 모니터링
- **Latency Analysis**: 각 모델 호출의 time-to-first-token 및 전체 응답 시간 추적
- **Error Monitoring**: 전체 컨텍스트와 함께 실패한 모델 호출, 도구 오류, 에이전트 예외 식별
- **Evaluations**: 지속적인 개선을 위해 트레이스에 품질 점수 및 사용자 지정 평가 연결

자세한 내용은 [Datadog LLM Observability 문서](https://docs.datadoghq.com/llm_observability/)를 참고하세요.

## 리소스 정리(선택 사항)

배포된 리소스를 정리합니다.

In [ ]:
import boto3

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)

ecr_client = boto3.client("ecr", region_name=region)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)

## 요약

Datadog LLM Observability가 적용된 Strands 에이전트를 Amazon Bedrock AgentCore Runtime에 성공적으로 배포했습니다. 이 구현에서는 다음 내용을 살펴보았습니다.
- 사용자 지정 관측성 provider를 사용하기 위해 AgentCore의 기본 ADOT 비활성화
- Datadog으로 트레이스를 직접 전송하도록 OpenTelemetry TracerProvider 구성
- `dd-otlp-source=llmobs`를 사용하여 트레이스를 Datadog LLM Observability로 라우팅
- LLM 전용 트레이스 화면을 위한 GenAI semantic convention 활성화
- AgentCore starter toolkit SDK를 통한 호출

### 참고 자료

- [AgentCore Observability docs](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html)
- [Datadog LLM Observability OTEL Instrumentation](https://docs.datadoghq.com/llm_observability/instrumentation/otel_instrumentation/)
- [Strands Agents Observability](https://strandsagents.com/latest/documentation/docs/user-guide/observability-evaluation/observability/)
- [OpenTelemetry GenAI Semantic Conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/)